# BrainScan AI — Classification d’images IRM cérébrales

**Objectif :** préparer un dataset d’images IRM, concevoir un réseau de neurones convolutif (CNN), l’entraîner, évaluer ses performances et sauvegarder le modèle.

Les classes sont `glioma`, `meningioma`, `notumor` et `pituitary`. Ce projet est expérimental : ses résultats ne constituent pas un diagnostic médical.

## Étape 0 — Configuration et importation des bibliothèques

Importer les bibliothèques nécessaires, définir les chemins du projet et fixer les graines aléatoires pour rendre les expériences plus reproductibles. Les dossiers de documentation et de modèles seront créés s’ils n’existent pas.

In [ ]:
import sys
import time
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

sys.path.append("..")

from src.data_loader import load_dataset
from src.preprocessing import encode_labels, split_data, normalize, one_hot
from src.model import build_model, save_architecture

SEED = 42
DATA_DIR = "../Data"

np.random.seed(SEED)
tf.random.set_seed(SEED)

Path("../docs").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

## Étape 1 — Charger et valider les images

Charger les images depuis le dossier du dataset. La fonction `load_dataset()` centralise le chargement et renvoie trois éléments : les images, leurs étiquettes et les noms des classes.

La validation doit notamment vérifier les extensions acceptées (`.jpeg`, `.jpg`, `.bmp`, `.png`), ignorer les fichiers non compatibles et gérer les erreurs de lecture. Ces contrôles doivent être implémentés dans `src/data_loader.py`. Ici, on vérifie la forme des tableaux obtenus et la correspondance entre images et étiquettes.

In [ ]:
images, labels, classes = load_dataset(DATA_DIR, seed=SEED)

print("Classes :", classes)
print("Images :", images.shape, images.dtype)
print("Étiquettes :", labels.shape, labels.dtype)

assert images.ndim == 4 and images.shape[1:] == (224, 224, 3)
assert len(images) == len(labels)
assert set(labels) == set(classes)

## Étape 2 — Explorer les données (EDA)

### 2.1 — Examiner la répartition des classes

Compter le nombre d’images de chaque classe et calculer le ratio entre la classe la plus représentée et la moins représentée. Un ratio proche de 1 indique une répartition relativement équilibrée. Le graphique est enregistré dans `docs/class_distribution.png`.

In [ ]:
counts = Counter(labels)
print("Images par classe :", counts)

imbalance_ratio = max(counts.values()) / min(counts.values())
print(f"Ratio de déséquilibre (max/min) : {imbalance_ratio:.2f}")

plt.figure(figsize=(7, 4))
plt.bar(counts.keys(), counts.values(), color="steelblue")
plt.title("Nombre d’images IRM par classe")
plt.ylabel("Nombre d’images")
plt.xlabel("Classe")
plt.savefig("../docs/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 — Afficher des exemples d’images

Afficher jusqu’à cinq images par classe pour observer la qualité visuelle, les variations de couleur et la diversité des images. L’apparence seule ne permet pas de conclure à un diagnostic.

In [ ]:
fig, plots = plt.subplots(len(classes), 5, figsize=(12, 2.5 * len(classes)))
plots = np.atleast_2d(plots)

for i, cls in enumerate(classes):
    idx = np.flatnonzero(labels == cls)[:5]
    for j, k in enumerate(idx):
        plots[i, j].imshow(images[k])
        plots[i, j].axis("off")
    plots[i, 0].set_title(cls, loc="left")

plt.tight_layout()
plt.savefig("../docs/samples_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3 — Décider si un rééquilibrage est nécessaire

Le dataset observé contient 1 621 images `glioma`, 1 645 `meningioma`, 2 000 `notumor` et 1 757 `pituitary`, soit un ratio d’environ 1,23. Le déséquilibre est modéré : on commence donc sans rééchantillonnage et on utilise une division stratifiée. On pourra envisager des poids de classes si les résultats de validation révèlent un problème propre à certaines classes.

Après l’affichage, noter ici les observations concernant la qualité et la variation des images, sans interpréter leur apparence comme un diagnostic.

## Étape 3 — Encoder les étiquettes, diviser les données et normaliser

Transformer les étiquettes textuelles en nombres avec `LabelEncoder`, puis diviser le dataset en ensembles d’entraînement, de validation et de test. La division doit être stratifiée pour préserver autant que possible la proportion de chaque classe.

Normaliser ensuite les pixels dans l’intervalle `[0, 1]` et convertir les étiquettes en vecteurs one-hot, adaptés à une classification multiclasse avec `categorical_crossentropy`.

In [ ]:
y, le = encode_labels(labels)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    images, y, seed=SEED
)

print("Formes des ensembles :", X_train.shape, X_val.shape, X_test.shape)

del images, labels, y

In [ ]:
X_train = normalize(X_train)
X_val = normalize(X_val)
X_test = normalize(X_test)

y_train = one_hot(y_train, len(le.classes_))
y_val = one_hot(y_val, len(le.classes_))
y_test = one_hot(y_test, len(le.classes_))

print(
    "Entraînement :", X_train.shape, X_train.dtype,
    float(X_train.min()), float(X_train.max())
)
print("Forme des étiquettes :", y_train.shape, y_train.dtype)

assert 0.0 <= X_train.min() and X_train.max() <= 1.0
assert len(X_train) == len(y_train)

## Étape 4 — Concevoir et inspecter le modèle CNN

Construire un réseau convolutif avec des couches `Conv2D` pour extraire des caractéristiques visuelles, `MaxPooling2D` pour réduire la taille des représentations, `Flatten` pour passer à un vecteur, `Dense` pour la classification et `Dropout` pour limiter le surapprentissage.

Les couches cachées utilisent généralement ReLU et la couche de sortie softmax, qui produit une probabilité pour chacune des quatre classes. L’optimiseur Adam, un taux d’apprentissage initial de `0.001` et la perte `categorical_crossentropy` sont des choix de départ à comparer sur les données de validation — pas des paramètres garantis optimaux.

In [ ]:
model = build_model(
    input_shape=X_train.shape[1:],
    num_classes=len(le.classes_),
    lr=1e-3
)

model.summary()

### 4.1 — Enregistrer le schéma de l’architecture

Créer une représentation visuelle du modèle pour inspecter l’enchaînement des couches. Si le diagramme ne peut pas être généré, vérifier que `pydot` et Graphviz sont installés.

In [ ]:
try:
    save_architecture(model, "../docs/model_architecture.png")
except Exception as exc:
    print(
        "Impossible de créer le diagramme. Vérifiez pydot et Graphviz :",
        exc
    )

## Étape 5 — Définir les hyperparamètres et entraîner le modèle

Définir le nombre maximal d’époques et la taille des lots (*batch size*). `ModelCheckpoint` sauvegarde le modèle qui obtient la meilleure perte de validation (`val_loss`), afin de ne pas dépendre uniquement des poids de la dernière époque. La bibliothèque `time` permet de mesurer la durée totale de l’entraînement.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32

checkpoint = ModelCheckpoint(
    filepath="../models/best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint]
)

train_time = time.time() - start
print(f"Durée d’entraînement : {train_time / 60:.1f} min")

### 5.1 — Visualiser les courbes d’apprentissage

Comparer les courbes d’accuracy et de loss sur l’entraînement et la validation. Un écart important entre les deux ensembles peut indiquer du surapprentissage. Une loss de validation qui augmente alors que la loss d’entraînement diminue peut également être un signal d’alerte.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history["accuracy"], label="Entraînement")
ax[0].plot(history.history["val_accuracy"], label="Validation")
ax[0].set_title("Accuracy")
ax[0].set_xlabel("Époque")
ax[0].legend()

ax[1].plot(history.history["loss"], label="Entraînement")
ax[1].plot(history.history["val_loss"], label="Validation")
ax[1].set_title("Loss")
ax[1].set_xlabel("Époque")
ax[1].legend()

plt.tight_layout()
plt.savefig("../docs/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Étape 6 — Évaluer le modèle sur l’ensemble de test

Charger le meilleur modèle sauvegardé par `ModelCheckpoint`, puis mesurer ses performances sur l’ensemble de test, qui n’a pas servi à ajuster les poids. L’accuracy donne une vue globale, tandis que la précision, le rappel et le F1-score détaillent les performances par classe.

In [ ]:
best_model = load_model("../models/best_model.keras")

test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss : {test_loss:.4f} | Test accuracy : {test_acc:.4f}")

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

### 6.1 — Générer la matrice de confusion et le rapport de classification

La matrice de confusion montre les classes réelles en lignes et les classes prédites en colonnes. Le rapport de classification présente la précision, le rappel et le F1-score pour chaque classe. Examiner en particulier les faux négatifs pour les classes associées à des tumeurs. Ces résultats décrivent uniquement ce dataset et cette séparation ; ils ne démontrent pas la sécurité clinique du modèle.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title("Matrice de confusion")
plt.tight_layout()
plt.savefig("../docs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=le.classes_,
        zero_division=0
    )
)

### 6.2 — Afficher des prédictions correctes et incorrectes

Afficher quelques images de test en indiquant leur classe réelle et la classe prédite. Ces exemples facilitent l’inspection qualitative des erreurs, mais ne remplacent pas les métriques quantitatives.

In [ ]:
def show_predictions(indices, title, n=6):
    selected = indices[:n]

    if len(selected) == 0:
        print(f"Aucun exemple à afficher : {title}")
        return

    fig, axes = plt.subplots(
        1, len(selected),
        figsize=(2.5 * len(selected), 3)
    )
    axes = np.atleast_1d(axes)

    for ax, k in zip(axes, selected):
        ax.imshow(X_test[k])
        ax.set_title(
            f"Réel : {le.classes_[y_true[k]]}\n"
            f"Prédit : {le.classes_[y_pred[k]]}",
            fontsize=8
        )
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()

    filename = title.lower().replace(" ", "_") + ".png"
    plt.savefig(f"../docs/{filename}", dpi=150, bbox_inches="tight")
    plt.show()


show_predictions(
    np.flatnonzero(y_pred == y_true),
    "Prédictions correctes"
)
show_predictions(
    np.flatnonzero(y_pred != y_true),
    "Prédictions incorrectes"
)

## Étape 7 — Sauvegarder le modèle final et les classes

Sauvegarder le meilleur modèle entraîné et le fichier contenant l’ordre des classes. Le fichier `classes.json` est indispensable pour interpréter correctement les indices de sortie lors de futures prédictions. Conserver ces fichiers ensemble.

In [ ]:
best_model.save("../models/final_model.keras")

with open("../models/classes.json", "w", encoding="utf-8") as f:
    json.dump(
        list(le.classes_),
        f,
        ensure_ascii=False,
        indent=2
    )

print("Modèle et noms des classes enregistrés.")

## Bilan et prochaines étapes

- Vérifier les courbes d’apprentissage et les métriques par classe.
- Documenter les confusions observées et les limites du dataset.
- Comparer les modèles ou hyperparamètres uniquement à partir des données de validation.
- Pour l’inférence, charger `final_model.keras` avec `classes.json` afin de convertir les indices prédits en noms de classes.
- La prédiction et l’interface Streamlit pourront ensuite être développées dans `src/` et `app/`.

**Limite importante :** ce prototype de classification d’images ne doit pas être utilisé seul pour prendre une décision médicale.